In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

In [16]:
# Importing dataset
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [17]:
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


In [18]:
for col in df:
    print(col, df[col].dtype)

categorial_cols = ['lead_source', 'industry', 'employment_status', 'location']
numerical_cols = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']

lead_source str
industry str
employment_status str
location str
annual_income float64
number_of_courses_viewed int64
interaction_count int64
lead_score float64
converted int64


In [19]:
"""
Check if the missing values are presented in the features.
If there are missing values:

    For categorical features, replace them with 'NA'
    For numerical features, replace with with 0.0
"""
print(df.isnull().sum())
for col in df:
    if df[col].dtype == 'str':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [20]:
print(df.isnull().sum())

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [21]:
# What is the most frequent observation (mode) for the column industry?
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [22]:
"""
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

    interaction_count and lead_score
    number_of_courses_viewed and lead_score
    number_of_courses_viewed and interaction_count
    annual_income and interaction_count

Only consider the pairs above when answering this question.
"""
print("interaction_count and lead_score:", df['interaction_count'].corr(df['lead_score']))
print("number_of_courses_viewed and lead_score:", df['number_of_courses_viewed'].corr(df['lead_score']))
print("number_of_courses_viewed and interaction_count:", df['number_of_courses_viewed'].corr(df['interaction_count']))
print("annual_income and interaction_count:", df['annual_income'].corr(df['interaction_count']))


interaction_count and lead_score: 0.9157457980418677
number_of_courses_viewed and lead_score: 0.7572042392104887
number_of_courses_viewed and interaction_count: 0.7216090038937009
annual_income and interaction_count: 0.12284235135955954


In [23]:
# Splitting the data and removing converted column
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train.converted
y_val = df_val.converted
y_test = df_test.converted

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [24]:
"""
    Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
    Round the scores to 2 decimals using round(score, 2).\

Which of these variables has the biggest mutual information score?

    industry
    location
    lead_source
    employment_status
"""
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

mi = df_full_train[categorial_cols].apply(mutual_info_converted_score)
round(mi.sort_values(ascending=False), 2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [25]:
"""
Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.

    To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

    0.55
    0.65
    0.75
    0.85
"""
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorial_cols + numerical_cols].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorial_cols + numerical_cols].to_dict(orient='records')
X_val = dv.transform(val_dict)


model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
print(round((y_val == converted_decision).mean(), 2))

main_acc = (y_val == converted_decision).mean()

0.64


In [26]:
"""
Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

    'lead_source'
    'number_of_courses_viewed'
    'interaction_count'

Note: The difference doesn't have to be positive.
"""
all_cols = categorial_cols + numerical_cols

for col in all_cols:
    selected_cols = all_cols.copy()
    df_train_sample = df_train.copy()
    df_val_sample = df_val.copy()
    del df_train_sample[col]
    del df_val_sample[col]
    selected_cols.remove(col)
    train_dict_sample = df_train_sample[selected_cols].to_dict(orient='records')
    X_train_sample = dv.fit_transform(train_dict_sample)
    
    val_dict_sample = df_val_sample[selected_cols].to_dict(orient='records')
    X_val_sample = dv.transform(val_dict_sample)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_sample, y_train)

    y_pred_sample = model.predict_proba(X_val_sample)[:, 1]
    converted_decision_sample = (y_pred_sample >= 0.5)
    print(col, main_acc - (y_val == converted_decision_sample).mean())

lead_source 0.0030000000000000027
industry 0.0
employment_status 0.0
location 0.0
annual_income -0.07899999999999996
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004
lead_score 0.0010000000000000009


In [30]:
"""
    Now let's train a regularized logistic regression.
    Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
    Train models using all the features as in Q4.
    Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

    0.000001
    0.00001
    0.0001
    0.001

    Note: If there are multiple options, select the smallest C.
"""
for reg_val in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=reg_val, max_iter=1000, random_state=42)
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    print(reg_val, round((y_val == converted_decision).mean(), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
